# HybridRecMCP — Synthetic Dataset Exploration
Independent, unofficial proof of concept developed by Nada Bastawi. All rows, prices and patterns are simulated.
This notebook explores the regenerated fixed 2023–2025 dataset. It is not evidence about a company or actual infestation rates.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load Data

Load the three main datasets from CSV files generated by the synthetic data generator.

In [ ]:
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
customers = pd.read_csv(ROOT / "data/customers.csv", dtype={"zip_code": str})
interactions = pd.read_csv(ROOT / "data/interactions.csv")
services = pd.read_csv(ROOT / "data/services.csv")
print({"customers": len(customers), "interactions": len(interactions), "services": len(services)})

## 2. Distribution Plots: Customer Demographics

Visualize the distribution of property types, ZIP codes, and building ages.

In [ ]:
# Join datasets for richer analysis
interactions_customers = interactions.merge(
    customers[["customer_id", "zip_code", "property_type", "building_age_years"]],
    on="customer_id",
    how="left",
)
interactions_full = interactions_customers.merge(
    services[["service_id", "category", "peak_season"]], on="service_id", how="left"
)
print("interactions_full columns:", interactions_full.columns.tolist())
print("sample service_name values:", interactions_full["service_name"].unique()[:5])

# Property type distribution
plt.figure(figsize=(12, 5))
ax = sns.countplot(
    data=customers,
    y="property_type",
    order=customers["property_type"].value_counts().index,
    palette="viridis",
    hue="property_type",
    legend=False,
)
ax.set_title("Customer Property Type Distribution")
ax.set_xlabel("Number of Customers")
ax.set_ylabel("Property Type")
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_width()):,}", (p.get_width() + 5, p.get_y() + p.get_height() / 2), va="center"
    )
plt.tight_layout()
plt.show()

# ZIP code distribution (top 10)
top_zips = customers["zip_code"].value_counts().nlargest(10)
plt.figure(figsize=(12, 5))
ax = sns.barplot(
    x=top_zips.values, y=top_zips.index, palette="rocket", hue=top_zips.index, legend=False
)
ax.set_title("Top 10 ZIP Codes by Customer Count")
ax.set_xlabel("Number of Customers")
ax.set_ylabel("ZIP Code")
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_width()):,}", (p.get_width() + 2, p.get_y() + p.get_height() / 2), va="center"
    )
plt.tight_layout()
plt.show()

# Building age distribution
plt.figure(figsize=(12, 5))
ax = sns.histplot(customers["building_age_years"], kde=True, color="#2a9d8f", bins=20)
ax.set_title("Distribution of Building Age (Years)")
ax.set_xlabel("Building Age (years)")
ax.set_ylabel("Customer Count")
plt.axvline(
    customers["building_age_years"].median(),
    color="darkred",
    linestyle="--",
    label=f"Median = {customers['building_age_years'].median():.0f}",
)
plt.legend()
plt.tight_layout()
plt.show()

### Interpretation
Inspect the computed plots above. Any pattern reflects generator assumptions, not verified company demand, retention or pest prevalence.


## 3. Service Trends and Seasonality

Analyze service popularity by season and highlight termite demand during spring and summer.

In [ ]:
season_service = (
    interactions_full.groupby(["season", "service_name"], as_index=False)
    .size()
    .rename(columns={"size": "service_count"})
)

plt.figure(figsize=(14, 7))
season_order = ["winter", "spring", "summer", "fall"]
termite_services = season_service[season_service["service_name"].str.contains("Termite")]
ax = sns.barplot(
    data=termite_services,
    x="season",
    y="service_count",
    hue="service_name",
    order=season_order,
    palette="magma",
)
ax.set_title("Termite Service Demand by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Number of Bookings")
for p in ax.patches:
    ax.annotate(
        int(p.get_height()),
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.legend(title="Service")
plt.tight_layout()
plt.show()

# Seasonal demand for all services
plt.figure(figsize=(16, 8))
ax = sns.barplot(
    data=season_service,
    x="season",
    y="service_count",
    hue="service_name",
    order=season_order,
    dodge=True,
)
ax.set_title("Service Demand by Season")
ax.set_xlabel("Season")
ax.set_ylabel("Total Bookings")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Interpretation
Inspect the computed plots above. Any pattern reflects generator assumptions, not verified company demand, retention or pest prevalence.


## 4. Ratings and Repeat Bookings

Evaluate rating distributions and repeat booking behavior across service categories.

In [ ]:
# Rating distribution per service
plt.figure(figsize=(16, 8))
service_order = (
    interactions_full.groupby("service_name")["rating"].median().sort_values(ascending=False).index
)
ax = sns.boxplot(
    data=interactions_full,
    x="rating",
    y="service_name",
    order=service_order,
    hue="service_name",
    legend=False,
    palette="coolwarm",
)
ax.set_title("Rating Distribution by Service")
ax.set_xlabel("Customer Rating")
ax.set_ylabel("Service")
plt.tight_layout()
plt.show()

# Repeat booking rate per service
repeat_rate = (
    interactions_full.groupby("service_name", as_index=False)["repeat_booking"]
    .mean()
    .sort_values("repeat_booking", ascending=False)
)
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=repeat_rate,
    x="repeat_booking",
    y="service_name",
    hue="service_name",
    legend=False,
    palette="cubehelix",
)
ax.set_title("Repeat Booking Rate by Service")
ax.set_xlabel("Repeat Booking Rate")
ax.set_ylabel("Service")
for p in ax.patches:
    ax.annotate(
        f"{p.get_width():.2f}", (p.get_width() + 0.01, p.get_y() + p.get_height() / 2), va="center"
    )
plt.tight_layout()
plt.show()

### Interpretation
Inspect the computed plots above. Any pattern reflects generator assumptions, not verified company demand, retention or pest prevalence.


## 5. Service Demand Heatmap by Property Type

Visualize which services are most commonly requested by each property type.

In [ ]:
heatmap_data = (
    interactions_full.groupby(["property_type", "service_name"])
    .size()
    .reset_index(name="count")
    .pivot(index="property_type", columns="service_name", values="count")
    .fillna(0)
)

plt.figure(figsize=(16, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label": "Booking Count"})
plt.title("Service Demand by Property Type")
plt.xlabel("Service")
plt.ylabel("Property Type")
plt.tight_layout()
plt.show()

### Interpretation
Inspect the computed plots above. Any pattern reflects generator assumptions, not verified company demand, retention or pest prevalence.
